# Log-probability descriptive analysis (isolated-judgment protocol)

Cross-checks `statistical_analysis/logprob_analysis.py` (a dependency-free, pure-stdlib
version run locally, without scipy/pandas/matplotlib available) using pandas/scipy/matplotlib
on the server. The two should produce identical `spearman_rho` and summary-stat values —
if they don't, that's a bug worth chasing down rather than trusting either blindly.

Covers the 4 models that currently have logprob data: Gemma-12B, Gemma-E4B, Qwen3-VL-4B,
Qwen3-VL-8B. Pixtral-12B and Mistral Small 3.1 24B are on hold pending the roster-exclusion
decision (see `docs/SESSION_HANDOFF.md`) — this notebook will pick them up automatically
(no code change needed) once/if their logprob files exist, since missing files are skipped.

Stays at the **descriptive layer only** by explicit decision (2026-08-03) — no significance
tests or corrections beyond the p-values `scipy.stats.spearmanr` returns for free. If a
formal confirmatory test is needed later, follow the pattern already used in
`gee_analysis.ipynb` (paired sign test) once there's a specific hypothesis to confirm rather
than explore.

## 0-1: Setup

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from IPython.display import display

ROOT_DIR = Path().resolve().parents[0]  # statistical_analysis/ -> repo root


In [ ]:
MODELS = {
    "gemma-12b":   ROOT_DIR / "experiments/e1/gemma4-12b/outputs",
    "gemma-e4b":   ROOT_DIR / "experiments/e1/gemma4-e4b/outputs",
    "qwen3-vl-4b": ROOT_DIR / "experiments/e1/qwen3-vl-4b/outputs",
    "qwen3-vl-8b": ROOT_DIR / "experiments/e1/qwen3-vl-8b/outputs",
    # Uncomment once/if these are approved and have logprob data:
    # "pixtral-12b":            ROOT_DIR / "experiments/e1/pixtral-12b/outputs",
    # "mistral-small-3.1-24b":  ROOT_DIR / "experiments/e1/mistral-small-3.1-24b/outputs",
}

CONDITIONS = {
    "baseline":         {"single": "e1_results_baseline_logprobs.json",         "yesno": "e1_results_baseline_yesno_logprobs.json"},
    "likes_only":       {"single": "e1_results_likes_only_logprobs.json",       "yesno": "e1_results_likes_only_yesno_logprobs.json"},
    "metrics":          {"single": "e1_results_metrics_logprobs.json",          "yesno": "e1_results_metrics_yesno_logprobs.json"},
    "likes_only_noise": {"single": "e1_results_likes_only_noise_logprobs.json", "yesno": "e1_results_likes_only_noise_yesno_logprobs.json"},
}
WORD = {"single": "like", "yesno": "yes"}


## 2: Load & consolidate

One row per trial, tagged with `model`/`condition`/`phrasing`. `p_like` is the
forced-choice softmax probability the model assigned to "like"/"yes" (see
`experiments/e1/e1_utils/logprob_scoring.py::extract_candidate_logprobs`), read directly
off the first generated token's distribution — not the greedy-decoded answer.

In [ ]:
def load_logprob_file(path, like_word):
    records = json.loads(path.read_text())
    rows = []
    for r in records:
        rows.append({
            "image": r["image"], "variant": r["variant"],
            "scale_value": r.get("scale_value"),
            "decoded_answer": r["answer"],
            "p_like": r["candidates"][like_word]["prob_forced_choice"],
        })
    return pd.DataFrame(rows)

frames = []
for model, outdir in MODELS.items():
    for condition, files in CONDITIONS.items():
        for phrasing, like_word in WORD.items():
            fpath = outdir / files[phrasing]
            if not fpath.exists():
                continue
            df = load_logprob_file(fpath, like_word)
            df["model"], df["condition"], df["phrasing"] = model, condition, phrasing
            frames.append(df)

combined = pd.concat(frames, ignore_index=True)
combined["decoded_like_binary"] = combined["decoded_answer"].isin(["like", "yes"]).astype(int)
print(f"Loaded {len(combined)} total trials.")

out_dir = ROOT_DIR / "statistical_analysis/outputs"
out_dir.mkdir(parents=True, exist_ok=True)
combined.to_csv(out_dir / "logprob_combined.csv", index=False)


## Section 2 — Saturation check

Is the decoded protocol's 0%/100% saturation genuine, or is it hiding real gradation
underneath? A saturated cell (`decoded_rate_pct` near 0 or 100) with a large `std_p_like`
means the binary decode was masking real variation — the model wasn't uniformly certain,
it just always landed on the same side of the threshold.

In [ ]:
summary = combined.groupby(["model", "condition", "phrasing"]).agg(
    n=("p_like", "size"),
    decoded_rate_pct=("decoded_like_binary", lambda s: round(s.mean() * 100, 1)),
    mean_p_like=("p_like", lambda s: round(s.mean(), 4)),
    std_p_like=("p_like", lambda s: round(s.std(), 4)),
    pct_confident=("p_like", lambda s: round(((s < 0.01) | (s > 0.99)).mean() * 100, 1)),
).reset_index()
display(summary)

summary.to_csv(out_dir / "logprob_saturation_summary.csv", index=False)


In [ ]:
flagged = summary[((summary.decoded_rate_pct >= 99) | (summary.decoded_rate_pct <= 1)) & (summary.std_p_like > 0.05)]
print("Saturated-but-not-really cells (decoded flat, std_p_like > 0.05):")
display(flagged)


## Section 3 — Does confidence track engagement disparity where decode doesn't move?

Spearman correlation between `scale_value` and `p_like`, computed separately per
model/condition/phrasing/variant. A model whose decoded answer never changes across scale
values can still show a real (positive or negative) `spearman_rho` here — that would mean
it's sensitive to engagement disparity internally even though that sensitivity never
crosses the threshold needed to flip its decision.

`p_value` is included since `scipy` returns it for free, but per the descriptive-layer
decision this isn't being used for any significance claim or correction here.

In [ ]:
scaled = combined[combined["scale_value"].notna()]
rows = []
for (model, condition, phrasing, variant), g in scaled.groupby(["model", "condition", "phrasing", "variant"]):
    if g["scale_value"].nunique() > 1:
        rho, p = spearmanr(g["scale_value"], g["p_like"])
        rows.append({"model": model, "condition": condition, "phrasing": phrasing,
                      "variant": variant, "n": len(g), "spearman_rho": round(rho, 3), "p_value": p})
corr_df = pd.DataFrame(rows)
display(corr_df)

corr_df.to_csv(out_dir / "logprob_scale_correlation.csv", index=False)


## Section 4 — Correctness-sensitivity gap: logprob-based vs. decoded-answer-based

`logprob_gap` = mean `p_like` for `correct`-variant images minus `incorrect`-variant
images. `decoded_gap_pct` is the same comparison using the binary decoded answer, in
percentage points, so the two columns are on different scales (probability vs. percentage
points) — compare them by direction and relative size, not raw magnitude.

In [ ]:
gap = combined.groupby(["model", "condition", "phrasing", "variant"])[["p_like", "decoded_like_binary"]].mean().unstack("variant")
gap["logprob_gap"] = gap[("p_like", "correct")] - gap[("p_like", "incorrect")]
gap["decoded_gap_pct"] = (gap[("decoded_like_binary", "correct")] - gap[("decoded_like_binary", "incorrect")]) * 100
gap_display = gap[["logprob_gap", "decoded_gap_pct"]].reset_index()
display(gap_display)

gap_display.to_csv(out_dir / "logprob_correctness_gap.csv", index=False)


## Section 5 — Distribution plots

Worth plotting first, per the local descriptive run (2026-08-03): Qwen3-VL-4B's `baseline`
condition decodes 100% "like" but is far from uniformly confident underneath: only ~3.5%
of trials are actually >0.99 or <0.01. Gemma-12B's `likes_only` condition never once
decodes to "like" (0% at every scale value), but still shows a real positive
`scale_value`-vs-`p_like` correlation for `correct`-variant images (ρ≈0.42 in the local
run) — the plot below should show that as a rightward density shift with scale, even
though every single trial still decodes to "scroll".

In [ ]:
def plot_p_like_distribution(df, model, condition, phrasing):
    sub = df[(df.model == model) & (df.condition == condition) & (df.phrasing == phrasing)]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(sub[sub.variant == "correct"]["p_like"], bins=30, alpha=0.6, label="correct")
    ax.hist(sub[sub.variant == "incorrect"]["p_like"], bins=30, alpha=0.6, label="incorrect")
    ax.set_xlabel("P(like/yes), forced-choice softmax")
    ax.set_ylabel("count")
    ax.set_title(f"{model} — {condition} ({phrasing})")
    ax.legend()
    plt.show()

plot_p_like_distribution(combined, "qwen3-vl-4b", "baseline", "single")


In [ ]:
plot_p_like_distribution(combined, "gemma-12b", "likes_only", "single")


In [ ]:
# Add more calls here for any other model/condition/phrasing worth a figure,
# e.g. plot_p_like_distribution(combined, "gemma-e4b", "likes_only_noise", "single")
